# End-to-End Pipeline: Nowcasting State-Level Unemployment Shocks

**Full pipeline in a single notebook:**

| Step | Source notebook | Output |
|------|----------------|--------|
| 1 | 01 — BLS LAUS | `laus_panel_with_shocks.parquet` |
| 2 | 02 — Google Trends | `google_trends_panel.parquet` |
| 3 | 03 — Macro & Leading Indicators | `fred_macro.parquet`, `state_initial_claims.parquet`, `state_payrolls.parquet` |
| 4 | 04 — Merge & Feature Engineering | `panel_with_features.parquet` |
| 5 | 04b — GT Expanding Z-scores | `panel_with_features.parquet` (overwritten) |
| 6 | 05b — Onset Rolling Forecast | `onset_predictions.parquet` |
| 7 | 06b — Evaluation | figures + tables |
| **BONUS** | **07 — Cost Model** | **decision-cost analysis, sensitivity figures, `07_cost_model_summary.csv`** |

> **Prerequisite — Google Trends raw CSVs**: Step 2 reads from `data/raw/google_trends/`.
> These files are collected by `src/collect_trends.py` and are **not** re-downloaded here.
> All other data sources (BLS, FRED, DOL) are downloaded automatically on first run and cached.

## Setup

Import all libraries used throughout the pipeline and define shared constants:
- **Paths**: raw data is cached under `data/raw/`; processed outputs go to `data/processed/` and `data/interim/`.
- **STATES_51**: the 50 states + DC, identified by their two-letter USPS abbreviations.
- **FIPS mappings**: BLS and CES data use numeric FIPS codes; we convert them to abbreviations on load.
- All HTTP requests include a `User-Agent` header with a contact email as required by BLS and FRED APIs.

In [1]:
import io, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from panelsplit.cross_validation import PanelSplit
from panelsplit.application import cross_val_fit_predict

warnings.filterwarnings("ignore", category=UserWarning)

RAW_BLS  = Path("../data/raw/bls_laus");  RAW_BLS.mkdir(parents=True, exist_ok=True)
RAW_GT   = Path("../data/raw/google_trends")
RAW_FRED = Path("../data/raw/fred");       RAW_FRED.mkdir(parents=True, exist_ok=True)
RAW_DOL  = Path("../data/raw/dol_eta539"); RAW_DOL.mkdir(parents=True, exist_ok=True)
RAW_CES  = Path("../data/raw/bls_ces");    RAW_CES.mkdir(parents=True, exist_ok=True)
INTERIM  = Path("../data/interim");        INTERIM.mkdir(parents=True, exist_ok=True)
PROC     = Path("../data/processed");      PROC.mkdir(parents=True, exist_ok=True)
FIG      = Path("../output/figures");      FIG.mkdir(parents=True, exist_ok=True)
TBL      = Path("../output/tables");       TBL.mkdir(parents=True, exist_ok=True)

BLS_HEADERS  = {"User-Agent": "academic-research felipemanzife@gmail.com"}
FRED_HEADERS = {"User-Agent": "academic-research felipemanzife@gmail.com"}
DATE_START, DATE_END = "2010-01-01", "2024-12-31"

STATES_51 = [
    "AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID",
    "IL","IN","IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO",
    "MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA",
    "RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY",
]
FIPS_TO_ABBREV_STR = {
    "01":"AL","02":"AK","04":"AZ","05":"AR","06":"CA","08":"CO","09":"CT",
    "10":"DE","11":"DC","12":"FL","13":"GA","15":"HI","16":"ID","17":"IL",
    "18":"IN","19":"IA","20":"KS","21":"KY","22":"LA","23":"ME","24":"MD",
    "25":"MA","26":"MI","27":"MN","28":"MS","29":"MO","30":"MT","31":"NE",
    "32":"NV","33":"NH","34":"NJ","35":"NM","36":"NY","37":"NC","38":"ND",
    "39":"OH","40":"OK","41":"OR","42":"PA","44":"RI","45":"SC","46":"SD",
    "47":"TN","48":"TX","49":"UT","50":"VT","51":"VA","53":"WA","54":"WV",
    "55":"WI","56":"WY",
}
FIPS_TO_ABBREV_INT = {int(k): v for k, v in FIPS_TO_ABBREV_STR.items()}

def strip_df(df):
    df.columns = df.columns.str.strip()
    return df.apply(lambda c: c.str.strip() if c.dtype == object else c)

print("Setup complete.")

Setup complete.


---
## Step 1 — BLS LAUS: Unemployment Rate Panel + Sahm Shocks

**Source**: [BLS Local Area Unemployment Statistics (LAUS)](https://www.bls.gov/lau/) — seasonally adjusted monthly unemployment rates for all 51 states (50 + DC), 2010–2024.

**Sahm rule**: a shock fires in state $s$ at month $t$ when the 3-month moving average of the UR rises ≥ 0.5 pp above its prior 12-month minimum. We code the shock as **only the first crossing month** (not every month the rule is satisfied), so each episode contributes exactly one positive observation.

Three thresholds (0.3, 0.5, 0.7 pp) are stored; the main forecasting target uses **threshold = 0.5** (the original Claudia Sahm specification).

**Output**: `laus_panel_with_shocks.parquet` — 9,180 rows (51 × 180 months).

In [2]:
def fetch_bls_laus(filename):
    local = RAW_BLS / filename
    if not local.exists():
        print(f"Downloading {filename}...")
        r = requests.get(f"https://download.bls.gov/pub/time.series/la/{filename}",
                         headers=BLS_HEADERS, timeout=120)
        r.raise_for_status()
        local.write_bytes(r.content)
        print(f"  saved {local.stat().st_size/1e6:.1f} MB")
    else:
        print(f"Using cached {filename}")
    return pd.read_csv(local, sep="\t", dtype=str)

raw_data   = strip_df(fetch_bls_laus("la.data.3.AllStatesS"))
raw_series = strip_df(fetch_bls_laus("la.series"))
raw_area   = strip_df(fetch_bls_laus("la.area"))

series_sa = raw_series[
    (raw_series["area_type_code"] == "A") &
    (raw_series["measure_code"]   == "03") &
    (raw_series["seasonal"]       == "S")
].copy()
series_sa["state_fips"] = series_sa["area_code"].str[2:4]
series_sa = series_sa[series_sa["state_fips"].isin(FIPS_TO_ABBREV_STR)].copy()
series_sa["state_code"] = series_sa["state_fips"].map(FIPS_TO_ABBREV_STR)
area_states = raw_area[raw_area["area_type_code"] == "A"][["area_code","area_text"]].copy()
area_states.columns = ["area_code","state_name"]
series_sa = series_sa.merge(area_states, on="area_code", how="left")

target_ids = set(series_sa["series_id"])
df = raw_data[raw_data["series_id"].isin(target_ids)].copy()
df = df.merge(series_sa[["series_id","state_code","state_name"]], on="series_id", how="left")
df["year"]  = df["year"].astype(int)
df["month"] = df["period"].str[1:].astype(int)
df = df[df["month"] <= 12].copy()
df["unemployment_rate"] = pd.to_numeric(df["value"], errors="coerce")
df = df.dropna(subset=["unemployment_rate"])
df["date"] = pd.to_datetime(df[["year","month"]].assign(day=1))
df = df[(df["date"] >= DATE_START) & (df["date"] <= DATE_END)]
laus = (df[["state_code","state_name","year","month","date","unemployment_rate"]]
        .sort_values(["state_code","date"]).reset_index(drop=True))

assert laus.groupby("state_code")["date"].nunique().eq(180).all(), "Unbalanced LAUS panel"
print(f"LAUS panel: {laus.shape}  |  51 states × 180 months ✓")

Using cached la.data.3.AllStatesS
Using cached la.series
Using cached la.area
LAUS panel: (9180, 6)  |  51 states × 180 months ✓


In [3]:
def build_sahm_shock(df, threshold=0.5, ma_window=3, lookback=12):
    shock_col = f"shock_sahm_{int(round(threshold*10)):02d}"
    results = []
    for _, g in df.groupby("state_code", sort=True):
        g = g.sort_values("date").copy()
        g["ma3"]      = g["unemployment_rate"].rolling(ma_window, min_periods=ma_window).mean()
        g["min12"]    = g["ma3"].shift(1).rolling(lookback, min_periods=lookback).min()
        g["sahm_gap"] = g["ma3"] - g["min12"]
        above         = (g["sahm_gap"] >= threshold).astype(int)
        g[shock_col]  = ((above == 1) & (above.shift(1).fillna(0).astype(int) == 0)).astype(int)
        results.append(g)
    return pd.concat(results, ignore_index=True).sort_values(["state_code","date"]).reset_index(drop=True)

df05 = build_sahm_shock(laus, 0.5)
for thr in [0.3, 0.7]:
    col = f"shock_sahm_{int(round(thr*10)):02d}"
    tmp = build_sahm_shock(laus, thr)[["state_code","date",col]]
    df05 = df05.merge(tmp, on=["state_code","date"], how="left")

panel_shocks = df05.drop(columns=["min12"])
panel_out = panel_shocks[[
    "state_code","state_name","year","month","date",
    "unemployment_rate","ma3","sahm_gap",
    "shock_sahm_03","shock_sahm_05","shock_sahm_07"
]].copy()
panel_out.to_parquet(INTERIM / "laus_panel_with_shocks.parquet", index=False)
for col, thr in [("shock_sahm_03",0.3),("shock_sahm_05",0.5),("shock_sahm_07",0.7)]:
    print(f"  threshold={thr}: {int(panel_out[col].sum())} shocks  ({panel_out[col].mean():.2%} base rate)")
print(f"Saved laus_panel_with_shocks.parquet  {panel_out.shape}")

  threshold=0.3: 138 shocks  (1.50% base rate)
  threshold=0.5: 101 shocks  (1.10% base rate)
  threshold=0.7: 73 shocks  (0.80% base rate)
Saved laus_panel_with_shocks.parquet  (9180, 11)


---
## Step 2 — Google Trends Panel

**Why Google Trends?** Search queries are available in real time with no publication lag, making them attractive as a nowcasting input. When people start searching more for "layoffs" or "file for unemployment", it often precedes the official unemployment figures by weeks.

**Terms used** (6): `unemployment`, `file for unemployment`, `unemployment benefits`, `jobs hiring`, `layoffs`, `resume`. Together they capture both distress signals (layoffs, claims) and active job-search behaviour (resume, jobs hiring).

**Data collection**: Google returns weekly relative indices (0–100, rescaled within each state-term series). The raw CSVs are collected once by `src/collect_trends.py` and cached locally — they are **not** re-downloaded here. We aggregate weekly data to calendar months by taking the mean.

> **Prerequisite**: Step 2 reads from `data/raw/google_trends/`. Run `src/collect_trends.py` first if the files are missing.

**Output**: `google_trends_panel.parquet` — 55,080 rows (51 × 6 terms × 180 months).

In [4]:
TERMS  = ["unemployment","file for unemployment","unemployment benefits",
          "jobs hiring","layoffs","resume"]
def slug(t): return t.replace(" ","_")
def cpath(s, t): return RAW_GT / f"{s}__{slug(t)}.csv"

chunks = []
for state in STATES_51:
    for term in TERMS:
        p = cpath(state, term)
        if p.exists() and p.stat().st_size > 100:
            df_gt = pd.read_csv(p, parse_dates=["week_start"])
            chunks.append(df_gt)

raw_gt = pd.concat(chunks, ignore_index=True)
raw_gt["week_start"] = pd.to_datetime(raw_gt["week_start"])
raw_gt["interest"]   = pd.to_numeric(raw_gt["interest"], errors="coerce")

gt_panel = (raw_gt.rename(columns={"week_start":"month"})
    .assign(year=lambda x: x["month"].dt.year,
            month_num=lambda x: x["month"].dt.month)
    [["state_code","term","year","month_num","month","interest"]]
    .sort_values(["state_code","term","month"]).reset_index(drop=True))
gt_panel = gt_panel[(gt_panel["month"] >= DATE_START) & (gt_panel["month"] <= DATE_END)]

assert abs(len(gt_panel) - 51*6*180) / (51*6*180) < 0.02, "GT panel size off by >2%"
print(f"GT panel: {gt_panel.shape}  ({gt_panel['state_code'].nunique()} states, {gt_panel['term'].nunique()} terms) ✓")
gt_panel.to_parquet(INTERIM / "google_trends_panel.parquet", index=False)
print("Saved google_trends_panel.parquet")

GT panel: (55080, 6)  (51 states, 6 terms) ✓
Saved google_trends_panel.parquet


---
## Step 3 — Macro Controls & Leading Indicators

Three data sources provide the national and state-level context features:

**FRED (national macro)**  
Downloaded via FRED's public CSV endpoint. Five series:
- `UNRATE` — national unemployment rate (coincident indicator, context for state-level moves)
- `ICSA` — weekly initial jobless claims (leading, high-frequency)
- `VIXCLS` — VIX implied volatility (financial stress proxy)
- `NASDAQCOM` — NASDAQ Composite log return (risk sentiment)
- `T10Y2Y` — 10Y–2Y Treasury spread (yield curve inversion → recession signal)

Weekly/daily series are collapsed to monthly means (or last, for NASDAQ).

**DOL ETA-539 (state initial claims)**  
Weekly state-level initial unemployment insurance claims from the Department of Labor. Summed to monthly totals and log-transformed in the feature step, giving a timely, high-frequency state labor-market signal.

**BLS CES (state nonfarm payrolls)**  
Seasonally adjusted total nonfarm employment from the Current Employment Statistics survey. Year-on-year growth rate used as a medium-frequency indicator of labor demand.

**Outputs**: `fred_macro.parquet`, `state_initial_claims.parquet`, `state_payrolls.parquet`.

In [5]:
def fetch_fred(series_id):
    local = RAW_FRED / f"{series_id}.csv"
    if not local.exists():
        r = requests.get(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}",
                         headers=FRED_HEADERS, timeout=30)
        r.raise_for_status()
        local.write_bytes(r.content)
    df_f = pd.read_csv(local)
    df_f.columns = ["date", series_id]
    df_f["date"] = pd.to_datetime(df_f["date"])
    df_f[series_id] = pd.to_numeric(df_f[series_id], errors="coerce")
    return df_f[(df_f["date"] >= DATE_START) & (df_f["date"] <= DATE_END)].reset_index(drop=True)

unrate   = fetch_fred("UNRATE").set_index("date")
icsa_w   = fetch_fred("ICSA").set_index("date")
icsa     = icsa_w.resample("MS").mean().rename(columns={"ICSA":"icsa_mean"})
vix_d    = fetch_fred("VIXCLS").set_index("date")
vix      = vix_d.resample("MS").mean().rename(columns={"VIXCLS":"vix_mean"})
ndq_d    = fetch_fred("NASDAQCOM").set_index("date")
ndq_m    = ndq_d.resample("MS").last()
ndq_ret  = np.log(ndq_m / ndq_m.shift(1)).rename(columns={"NASDAQCOM":"nasdaq_logret"})
t10y2y_d = fetch_fred("T10Y2Y").set_index("date")
t10y2y   = t10y2y_d.resample("MS").mean().rename(columns={"T10Y2Y":"t10y2y_mean"})

fred_macro = (unrate.join(icsa).join(vix).join(ndq_ret).join(t10y2y)
              .rename(columns={"UNRATE":"unrate_nat"})
              .reset_index().rename(columns={"date":"month"}))
fred_macro = fred_macro[(fred_macro["month"] >= DATE_START) & (fred_macro["month"] <= DATE_END)]
fred_macro.to_parquet(INTERIM / "fred_macro.parquet", index=False)
print(f"FRED macro: {fred_macro.shape}  |  NaNs: {fred_macro.isna().sum().sum()}")

# DOL ETA-539: state initial claims
local_dol = RAW_DOL / "ar539.csv"
if not local_dol.exists():
    print("Downloading ar539.csv (15 MB)...")
    r = requests.get("https://oui.doleta.gov/unemploy/csv/ar539.csv",
                     headers=BLS_HEADERS, timeout=120)
    r.raise_for_status()
    local_dol.write_bytes(r.content)
raw_dol = pd.read_csv(local_dol, low_memory=False, usecols=["st","rptdate","c3"])
raw_dol["rptdate"] = pd.to_datetime(raw_dol["rptdate"], errors="coerce")
raw_dol["c3"]      = pd.to_numeric(raw_dol["c3"], errors="coerce")
raw_dol = raw_dol.rename(columns={"st":"state_code","rptdate":"week_end","c3":"init_claims"})
raw_dol = raw_dol[raw_dol["state_code"].isin(STATES_51) &
                  (raw_dol["week_end"] >= DATE_START) &
                  (raw_dol["week_end"] <= DATE_END)].copy()
raw_dol["month"] = raw_dol["week_end"].dt.to_period("M").dt.to_timestamp()
state_claims = (raw_dol.groupby(["state_code","month"], as_index=False)["init_claims"]
                .sum().rename(columns={"init_claims":"init_claims_monthly"})
                .sort_values(["state_code","month"]).reset_index(drop=True))
state_claims.to_parquet(INTERIM / "state_initial_claims.parquet", index=False)
print(f"State claims: {state_claims.shape}")

# BLS CES: state nonfarm payrolls
def fetch_bls_ces(filename):
    local = RAW_CES / filename
    if not local.exists():
        print(f"  Downloading {filename}...")
        r = requests.get(f"https://download.bls.gov/pub/time.series/sm/{filename}",
                         headers=BLS_HEADERS, timeout=120)
        r.raise_for_status()
        local.write_bytes(r.content)
    return local.read_bytes()

sm_series = strip_df(pd.read_csv(io.BytesIO(fetch_bls_ces("sm.series")), sep="\t", dtype=str))
for col in ["state_code","area_code","supersector_code","data_type_code"]:
    sm_series[col] = pd.to_numeric(sm_series[col], errors="coerce")
series_meta = sm_series[
    (sm_series["seasonal"] == "S") & (sm_series["area_code"] == 0) &
    (sm_series["supersector_code"] == 0) & (sm_series["data_type_code"] == 1) &
    (sm_series["state_code"].isin(FIPS_TO_ABBREV_INT))
].copy()
series_meta["state_abbrev"] = series_meta["state_code"].map(FIPS_TO_ABBREV_INT)
target_ids_ces = set(series_meta["series_id"])

sm_data = strip_df(pd.read_csv(io.BytesIO(fetch_bls_ces("sm.data.55.TotalNonFarmStateWide.All")),
                                sep="\t", dtype=str))
sm_data = sm_data[sm_data["series_id"].isin(target_ids_ces)].copy()
sm_data["year"]      = pd.to_numeric(sm_data["year"], errors="coerce").astype("Int64")
sm_data["month_num"] = pd.to_numeric(sm_data["period"].str[1:], errors="coerce").astype("Int64")
sm_data["value"]     = pd.to_numeric(sm_data["value"], errors="coerce")
sm_data = sm_data[sm_data["month_num"] <= 12].dropna(subset=["value"])
sm_data["month"] = pd.to_datetime(sm_data[["year","month_num"]].rename(columns={"month_num":"month"}).assign(day=1))
sm_data = sm_data[(sm_data["month"] >= DATE_START) & (sm_data["month"] <= DATE_END)]
sm_data = sm_data.merge(series_meta[["series_id","state_abbrev"]], on="series_id", how="left")
payrolls = (sm_data[["state_abbrev","month","value"]]
            .rename(columns={"state_abbrev":"state_code","value":"payrolls_thousands"})
            .sort_values(["state_code","month"]).reset_index(drop=True))
payrolls.to_parquet(INTERIM / "state_payrolls.parquet", index=False)
print(f"State payrolls: {payrolls.shape}")
print("\nStep 3 complete ✓")

FRED macro: (180, 6)  |  NaNs: 1


State claims: (9180, 3)


State payrolls: (9180, 3)

Step 3 complete ✓


---
## Step 4 — Merge & Feature Engineering

All interim files are joined into a single balanced panel (51 states × ~165 months after NaN trimming). Features are organized in four groups:

| Group | Variables | Rationale |
|-------|-----------|-----------|
| **A — Google Trends** | `gt_*_z`, `gt_*_d1`, `gt_*_std3`, `gt_*_yoy` | Real-time search signal; z-score, 1-month diff, 3-month rolling std, and YoY growth |
| **B — UR lags** | `ur_lag1/2/3/6/12`, `ur_chg1/3`, `ur_std3` | Autoregressive state of the unemployment cycle |
| **C — Since-vars** | `months_since_last_shock`, `cumulative_shocks_24m`, `sahm_gap_lag1` | Position in the shock cycle; how long since the last episode |
| **D — Macro lags** | FRED series at lag 1, `log_claims_lag1`, `payrolls_yoy_lag1` | National financial/macro conditions and state-level leading indicators |

**Publication-lag discipline**: all official statistics (UR, claims, payrolls, FRED) are lagged by at least one month so no information unavailable at the forecast origin leaks into the features.

> **Note**: the z-scores in Group A still use full-panel mean/std here — Step 5 immediately overwrites them with look-ahead-safe expanding versions.

**Output**: `panel_with_features.parquet` — ~8,400 rows after dropping rows where any feature is NaN (early months lack enough history for rolling windows).

In [6]:
laus_feat    = pd.read_parquet(INTERIM / "laus_panel_with_shocks.parquet").rename(columns={"month":"month_of_year"})
trends_long  = pd.read_parquet(INTERIM / "google_trends_panel.parquet").rename(columns={"month":"date"})
fred_feat    = pd.read_parquet(INTERIM / "fred_macro.parquet").rename(columns={"month":"date"})
claims_feat  = pd.read_parquet(INTERIM / "state_initial_claims.parquet").rename(columns={"month":"date"})
pay_feat     = pd.read_parquet(INTERIM / "state_payrolls.parquet").rename(columns={"month":"date"})

terms_f    = sorted(trends_long["term"].unique())
term_slugs = {t: "gt_" + t.replace(" ","_") for t in terms_f}
gt_cols    = list(term_slugs.values())

trends_wide = (trends_long
    .pivot_table(index=["state_code","date"], columns="term", values="interest", aggfunc="mean")
    .reset_index())
trends_wide.columns = ["state_code","date"] + [term_slugs[t] for t in terms_f]

panel = (laus_feat
    .merge(trends_wide, on=["state_code","date"], how="left")
    .merge(fred_feat,   on="date",                how="left")
    .merge(claims_feat, on=["state_code","date"], how="left")
    .merge(pay_feat,    on=["state_code","date"], how="left")
    .sort_values(["state_code","date"]).reset_index(drop=True))

print(f"Panel after merge: {panel.shape}")

Panel after merge: (9180, 24)


In [7]:
# ── Group A: Google Trends (global z-score — overwritten by Step 5) ──────────
feature_cols_A = []
for col in gt_cols:
    panel[f"{col}_z"]    = panel.groupby("state_code")[col].transform(
        lambda x: (x - x.mean()) / max(x.std(ddof=0), 1e-8))
    panel[f"{col}_d1"]   = panel.groupby("state_code")[f"{col}_z"].transform(lambda x: x.diff(1))
    panel[f"{col}_std3"] = panel.groupby("state_code")[f"{col}_z"].transform(
        lambda x: x.rolling(3, min_periods=2).std())
    panel[f"{col}_yoy"]  = panel.groupby("state_code")[col].transform(
        lambda x: (x + 1) / (x.shift(12) + 1) - 1)
    feature_cols_A += [f"{col}_z", f"{col}_d1", f"{col}_std3", f"{col}_yoy"]

# ── Group B: UR lags ───────────────────────────────────────────────────────────
for lag in [1, 2, 3, 6, 12]:
    panel[f"ur_lag{lag}"] = panel.groupby("state_code")["unemployment_rate"].transform(
        lambda x, l=lag: x.shift(l))
panel["ur_chg1"] = panel["ur_lag1"] - panel["ur_lag2"]
panel["ur_chg3"] = panel["ur_lag1"] - panel["ur_lag3"]
panel["ur_std3"] = panel.groupby("state_code")["unemployment_rate"].transform(
    lambda x: x.shift(1).rolling(3, min_periods=2).std())
feature_cols_B = [f"ur_lag{l}" for l in [1,2,3,6,12]] + ["ur_chg1","ur_chg3","ur_std3"]

# ── Group C: Since-variables ───────────────────────────────────────────────────
def months_since_last_shock(s):
    result = np.full(len(s), 36.0)
    last_pos = None
    for i, v in enumerate(s.values):
        if last_pos is not None:
            result[i] = min(i - last_pos, 36)
        if v == 1:
            last_pos = i
    return pd.Series(result, index=s.index)

panel["months_since_last_sahm_shock"] = panel.groupby("state_code")["shock_sahm_05"].transform(months_since_last_shock)
panel["cumulative_sahm_shocks_24m"]   = panel.groupby("state_code")["shock_sahm_05"].transform(
    lambda x: x.shift(1).rolling(24, min_periods=1).sum())
panel["sahm_gap_lag1"] = panel.groupby("state_code")["sahm_gap"].transform(lambda x: x.shift(1))
feature_cols_C = ["months_since_last_sahm_shock","cumulative_sahm_shocks_24m","sahm_gap_lag1"]

# ── Group D: Macro lags ────────────────────────────────────────────────────────
fred_cols = ["unrate_nat","icsa_mean","vix_mean","nasdaq_logret","t10y2y_mean"]
for col in fred_cols:
    panel[f"{col}_lag1"] = panel.groupby("state_code")[col].transform(lambda x, c=col: x.shift(1))
panel["log_claims"]      = np.log(panel["init_claims_monthly"].replace(0, np.nan))
panel["log_claims_lag1"] = panel.groupby("state_code")["log_claims"].transform(lambda x: x.shift(1))
panel["payrolls_yoy"]    = panel.groupby("state_code")["payrolls_thousands"].transform(lambda x: x / x.shift(12) - 1)
panel["payrolls_yoy_lag1"] = panel.groupby("state_code")["payrolls_yoy"].transform(lambda x: x.shift(1))
feature_cols_D = [f"{c}_lag1" for c in fred_cols] + ["log_claims_lag1","payrolls_yoy_lag1"]

# ── Assemble & save ────────────────────────────────────────────────────────────
id_cols      = ["state_code","state_name","date","year","month_of_year"]
target_cols  = ["shock_sahm_03","shock_sahm_05","shock_sahm_07"]
all_features = feature_cols_A + feature_cols_B + feature_cols_C + feature_cols_D + ["state_code","month_of_year"]
keep_cols    = list(dict.fromkeys(id_cols + target_cols + all_features))
panel_out    = panel[keep_cols].copy()

n_before = len(panel_out)
panel_out = panel_out.dropna(subset=[c for c in keep_cols if c not in id_cols + target_cols + ["state_name"]])
panel_out.to_parquet(PROC / "panel_with_features.parquet", index=False)
print(f"Dropped {n_before - len(panel_out)} rows with NaN features  ({100*(n_before-len(panel_out))/n_before:.1f}%)")
print(f"Saved panel_with_features.parquet  {panel_out.shape}")

Dropped 765 rows with NaN features  (8.3%)
Saved panel_with_features.parquet  (8415, 50)


---
## Step 5 — GT Expanding Z-scores (fix look-ahead bias)

Step 4 standardized Google Trends using the **full-panel mean and std** for each state-term pair. This introduces look-ahead bias: the mean and std are computed over 2010–2024, so a pre-COVID observation in 2015 is effectively normalized using knowledge of the 2020 spike, which a forecaster in 2015 would not have had.

**Fix**: replace `gt_*_z`, `gt_*_d1`, and `gt_*_std3` with **expanding-window** versions. At time $T$, the mean and std are computed from months $\{1, 2, \ldots, T-1\}$ only (`shift(1).expanding()`). This ensures the z-score at any point in time is computed solely from past data, as it would have been in real time.

The `yoy` feature is unaffected (it is already a ratio relative to the same calendar month one year prior, with no cross-time pooling).

**Output**: `panel_with_features.parquet` overwritten in place — same shape, cleaner features.

In [8]:
# Load full 9180-row GT panel (no NaN filtering) to compute expanding z-scores
trends_long_full = pd.read_parquet(INTERIM / "google_trends_panel.parquet").rename(columns={"month":"date"})
terms_f2  = sorted(trends_long_full["term"].unique())
term_slugs2 = {t: "gt_" + t.replace(" ","_") for t in terms_f2}
gt_cols2  = list(term_slugs2.values())

# Build full anchor panel (9180 rows) aligned to LAUS dates
laus_anchor = pd.read_parquet(INTERIM / "laus_panel_with_shocks.parquet")[["state_code","date"]]
trends_wide_full = (trends_long_full
    .pivot_table(index=["state_code","date"], columns="term", values="interest", aggfunc="mean")
    .reset_index())
trends_wide_full.columns = ["state_code","date"] + gt_cols2
full_gt = laus_anchor.merge(trends_wide_full, on=["state_code","date"], how="left")
full_gt = full_gt.sort_values(["state_code","date"]).reset_index(drop=True)

# Expanding z-scores: at time T use only T-1, T-2, ... history
gt_z_cols = []
for col in gt_cols2:
    exp_mean = full_gt.groupby("state_code")[col].transform(lambda x: x.shift(1).expanding().mean())
    exp_std  = full_gt.groupby("state_code")[col].transform(
        lambda x: x.shift(1).expanding().std().clip(lower=1e-8))
    full_gt[f"{col}_z"]    = (full_gt[col] - exp_mean) / exp_std
    full_gt[f"{col}_d1"]   = full_gt.groupby("state_code")[f"{col}_z"].transform(lambda x: x.diff(1))
    full_gt[f"{col}_std3"] = full_gt.groupby("state_code")[f"{col}_z"].transform(
        lambda x: x.rolling(3, min_periods=2).std())
    gt_z_cols += [f"{col}_z", f"{col}_d1", f"{col}_std3"]

# Overwrite panel_with_features.parquet
panel_feat = pd.read_parquet(PROC / "panel_with_features.parquet")
panel_feat = panel_feat.drop(columns=gt_z_cols)
new_gt     = full_gt[["state_code","date"] + gt_z_cols]
panel_feat = panel_feat.merge(new_gt, on=["state_code","date"], how="left")

n_before = len(panel_feat)
check_cols = [c for c in panel_feat.columns
              if c not in id_cols + target_cols + ["state_name"]]
panel_feat = panel_feat.dropna(subset=check_cols)
panel_feat.to_parquet(PROC / "panel_with_features.parquet", index=False)
print(f"Expanding z-scores applied. Dropped {n_before - len(panel_feat)} additional rows.")
print(f"Saved panel_with_features.parquet  {panel_feat.shape}")

Expanding z-scores applied. Dropped 0 additional rows.
Saved panel_with_features.parquet  (8415, 50)


---
## Step 6 — Onset Rolling Forecast (05b)

**Target** — *onset*: `onset = 1` at month $T$ if the state is currently healthy (`sahm_shock[T] = 0`) and a Sahm shock begins in the next `h = 3` months (i.e. `max(sahm_shock[T+1:T+4]) = 1`). States already in shock are assigned `NaN` and excluded from both training and evaluation — predicting "will a shock start" is only meaningful when none is currently underway.

This is harder than predicting the shock itself: we need to anticipate the *start* of a deterioration up to three months ahead, not just confirm one is happening.

**Rolling CV design**:
- `n_splits = 36`, `test_size = 1`, `gap = 2` — OOS window 2022-01 → 2024-12
- `gap = 2` rather than `gap = 1`: because the onset label at $T$ looks forward to $\{T+1, T+2, T+3\}$, we must keep training labels at least 2 periods away from the test window to avoid label overlap
- `drop_na_in_y = True` in `cross_val_fit_predict` handles the in-shock NaN rows

**Models**: `RandomForestClassifier(class_weight="balanced")` with Ben's session_3 hyperparameters (`max_depth=5`, `max_features=0.2`, `min_samples_leaf=15`). M4 wraps the RF in a pipeline that applies PCA(1) to the 6 GT z-score inputs, trained on training data only.

**Output**: `onset_predictions.parquet` — 1,836 rows with predicted probabilities for M0/M2/M3/M4.

In [9]:
# ── TargetEngineer ────────────────────────────────────────────────────────────
class TargetEngineer:
    def __init__(self, df, unit, time, y_col):
        self.df, self.unit, self.time, self.y_col = df.copy(), unit, time, y_col

    def any(self, threshold):
        any_col = f"any{self.y_col}_th{threshold}"
        self.df[any_col] = (self.df[self.y_col] > threshold).astype(int)
        return self.df.copy(), any_col

    def onset(self, threshold, horizon):
        df, any_col = self.any(threshold)
        def _onset(x, h):
            x_list = list(x); y = []
            for i in range(len(x_list)):
                if i + h < len(x_list) and x_list[i] == 0:
                    y.append(np.max(x_list[i+1:i+1+h]))
                else:
                    y.append(np.nan)
            return pd.Series(y, index=x.index)
        target_col = f"ons_{any_col}_h{horizon}"
        df[target_col] = self.df.groupby(self.unit)[any_col].transform(lambda x: _onset(x, horizon))
        return df[[self.y_col, any_col, target_col]]

# ── Load data ──────────────────────────────────────────────────────────────────
laus_raw = pd.read_parquet(INTERIM / "laus_panel_with_shocks.parquet")
laus_raw = laus_raw.sort_values(["state_code","date"]).reset_index(drop=True)
laus_raw["sahm_shock"] = (laus_raw["sahm_gap"] >= 0.5).astype(int)
laus_raw = laus_raw.set_index(["state_code","date"]).sort_index()

panel_feat2 = pd.read_parquet(PROC / "panel_with_features.parquet")

# ── Build onset target ─────────────────────────────────────────────────────────
threshold, horizon = 0, 3
te_ons = TargetEngineer(df=laus_raw, unit="state_code", time="date", y_col="sahm_shock")
ons_df = te_ons.onset(threshold=threshold, horizon=horizon)
ons_target_col = f"ons_anysahm_shock_th{threshold}_h{horizon}"
print(f"Onset target: {ons_target_col}")
print(f"  class-1 rate = {ons_df[ons_target_col].mean():.4f}")
print(f"  NaNs         = {ons_df[ons_target_col].isna().sum()}")

Onset target: ons_anysahm_shock_th0_h3
  class-1 rate = 0.0366
  NaNs         = 1097


In [10]:
# ── Feature encoding ──────────────────────────────────────────────────────────
state_dummies = pd.get_dummies(panel_feat2["state_code"], prefix="state", dtype=int)
month_dummies = pd.get_dummies(panel_feat2["month_of_year"], prefix="moy",   dtype=int)
panel_enc2 = pd.concat([panel_feat2, state_dummies, month_dummies], axis=1)
panel_enc2 = panel_enc2.set_index(["state_code","date"]).sort_index()

STATE_COLS = state_dummies.columns.tolist()
MOY_COLS   = month_dummies.columns.tolist()

# ── Align onset targets to features ───────────────────────────────────────────
since_col     = "months_since_last_sahm_shock"
ons_df_aligned = ons_df.reindex(panel_enc2.index)
ons_labels_df  = ons_df_aligned.merge(panel_enc2[[since_col]], left_index=True, right_index=True, how="left")

# ── Feature sets ───────────────────────────────────────────────────────────────
GT_TERMS       = ["file_for_unemployment","jobs_hiring","layoffs","resume","unemployment","unemployment_benefits"]
GT_FEATURES    = [f"gt_{t}_{s}" for t in GT_TERMS for s in ["z","d1","std3","yoy"]]
PCA_INPUT_COLS = [f"gt_{t}_z" for t in GT_TERMS]

AR_LAGS        = ["ur_lag2","ur_lag3","ur_lag6","ur_lag12","ur_chg1","ur_chg3","ur_std3"]
SINCE_VARS     = ["months_since_last_sahm_shock","cumulative_sahm_shocks_24m","sahm_gap_lag1"]
FRED_LAGS      = ["unrate_nat_lag1","icsa_mean_lag1","vix_mean_lag1","nasdaq_logret_lag1","t10y2y_mean_lag1"]
STATE_LEADING  = ["log_claims_lag1","payrolls_yoy_lag1"]

FEATURES_M0 = ["ur_lag1"] + STATE_COLS + MOY_COLS
FEATURES_M2 = FEATURES_M0 + AR_LAGS + SINCE_VARS + FRED_LAGS
FEATURES_M3 = FEATURES_M2 + STATE_LEADING
FEATURES_M4 = FEATURES_M3 + GT_FEATURES

for name, f in [("M0",FEATURES_M0),("M2",FEATURES_M2),("M3",FEATURES_M3),("M4",FEATURES_M4)]:
    print(f"  {name}: {len(f)} features")

  M0: 64 features
  M2: 79 features
  M3: 81 features
  M4: 105 features


In [11]:
# ── PanelSplit + model factory ────────────────────────────────────────────────
n_splits, test_size, gap = 36, 1, horizon - 1
ons_ps = PanelSplit(periods=panel_enc2.index.get_level_values("date"),
                    n_splits=n_splits, test_size=test_size, gap=gap)

def make_rf():
    return RandomForestClassifier(max_depth=5, max_features=0.2, min_samples_leaf=15,
                                  class_weight="balanced", random_state=200)

def make_m4_pipeline():
    pca_idx  = [FEATURES_M4.index(c) for c in PCA_INPUT_COLS]
    pass_idx = [i for i in range(len(FEATURES_M4)) if i not in pca_idx]
    prep = ColumnTransformer([
        ("pca", make_pipeline(SimpleImputer(strategy="constant", fill_value=0), PCA(n_components=1)), pca_idx),
        ("pass", "passthrough", pass_idx),
    ])
    return Pipeline([("prep", prep), ("rf", make_rf())])

MODELS = {"M0":(FEATURES_M0, make_rf()), "M2":(FEATURES_M2, make_rf()),
          "M3":(FEATURES_M3, make_rf()), "M4":(FEATURES_M4, make_m4_pipeline())}

# ── Rolling forecast ───────────────────────────────────────────────────────────
ons_final_preds = ons_ps.gen_test_labels(ons_labels_df)
print("Running onset rolling forecast (n_splits=36, gap=2)...")
for model_name, (feats, estimator) in MODELS.items():
    t0 = time.time()
    preds_arr, _ = cross_val_fit_predict(
        estimator=estimator, X=panel_enc2[feats], y=ons_df_aligned[ons_target_col],
        cv=ons_ps, method="predict_proba", drop_na_in_y=True)
    ons_final_preds[f"ons_preds_{model_name}"] = preds_arr[:, 1]
    print(f"  {model_name} done in {time.time()-t0:.1f}s")

ons_final_preds.to_parquet(PROC / "onset_predictions.parquet")
print(f"\nSaved onset_predictions.parquet  {ons_final_preds.shape}")

sub = ons_final_preds.dropna(subset=[ons_target_col])
print(f"\n{'Model':<6}  {'AUC-ROC':>9}  {'AUC-PR':>9}  {'Positives':>10}  {'N':>6}")
print("-"*46)
for m in ["M0","M2","M3","M4"]:
    col   = f"ons_preds_{m}"
    valid = sub.dropna(subset=[col])
    print(f"{m:<6}  {roc_auc_score(valid[ons_target_col], valid[col]):9.4f}  "
          f"{average_precision_score(valid[ons_target_col], valid[col]):9.4f}  "
          f"{int(valid[ons_target_col].sum()):10d}  {len(valid):6d}")
print(f"No-skill AUC-PR baseline: {sub[ons_target_col].mean():.4f}")

Running onset rolling forecast (n_splits=36, gap=2)...


  M0 done in 3.4s


  M2 done in 6.8s


  M3 done in 8.5s


  M4 done in 20.8s

Saved onset_predictions.parquet  (1836, 8)

Model     AUC-ROC     AUC-PR   Positives       N
----------------------------------------------
M0         0.5274     0.0733         104    1447
M2         0.8354     0.2930         104    1447
M3         0.8397     0.2923         104    1447
M4         0.8460     0.3239         104    1447
No-skill AUC-PR baseline: 0.0719


---
## Step 7 — Evaluation (06b)

**Why multiple metrics?** With only ~7% positive rate, accuracy is uninformative (a classifier that predicts "never" gets 93%). We report:

- **AUC-ROC**: threshold-free ranking quality; useful for comparing models but less sensitive to imbalance
- **AUC-PR** (average precision): penalises models that score non-events highly; more diagnostic for rare events
- **MCC / F1 / Sensitivity / Specificity**: at a fixed threshold chosen by **Youden's J** (= argmax TPR − FPR), balancing false alarms against missed detections
- **Brier Skill Score**: compares the model's mean squared probability error against a no-skill constant forecast; negative values mean the model is *worse* than just predicting the base rate (common for well-calibrated but uncertain problems)

**ROC and PR curves** visualize the full sensitivity/precision tradeoff across thresholds for all four models.

**Output**: `output/figures/e2e_roc_pr.png`, `output/tables/e2e_metrics.csv`.

In [12]:
from sklearn.metrics import (roc_curve, precision_recall_curve,
    f1_score, matthews_corrcoef, brier_score_loss, balanced_accuracy_score,
    confusion_matrix, log_loss)

MODEL_COLS = {"M0":"#888888","M2":"#2171b5","M3":"#41ab5d","M4":"#d7301f"}
MODEL_DASH = {"M0":(4,2),"M2":(6,2),"M3":(3,2),"M4":"solid"}

raw_preds = pd.read_parquet(PROC / "onset_predictions.parquet").reset_index()
raw_preds["year"] = raw_preds["date"].dt.year
TARGET    = ons_target_col
eval_df   = raw_preds.dropna(subset=[TARGET]).copy()
eval_df[TARGET] = eval_df[TARGET].astype(int)
base_rate = eval_df[TARGET].mean()
n_pos, n_total = int(eval_df[TARGET].sum()), len(eval_df)
print(f"Evaluation pool: {n_total:,} rows | {n_pos} onset events | {base_rate:.2%} base rate")
print(f"Date range: {eval_df['date'].min().strftime('%Y-%m')} → {eval_df['date'].max().strftime('%Y-%m')}")

Evaluation pool: 1,447 rows | 104 onset events | 7.19% base rate
Date range: 2022-01 → 2024-09


In [ ]:
# ── ROC + PR curves ───────────────────────────────────────────────────────────
MODELS_EVAL = ["M0","M2","M3","M4"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for m in MODELS_EVAL:
    col = f"ons_preds_{m}"
    ls  = MODEL_DASH[m] if isinstance(MODEL_DASH[m], str) else "--"
    lw  = 2.2 if m == "M4" else 1.6

    fpr, tpr, _ = roc_curve(eval_df[TARGET], eval_df[col])
    roc_auc = roc_auc_score(eval_df[TARGET], eval_df[col])
    axes[0].plot(fpr, tpr, color=MODEL_COLS[m], lw=lw, linestyle=ls, label=f"{m}  AUC={roc_auc:.3f}")

    prec, rec, _ = precision_recall_curve(eval_df[TARGET], eval_df[col])
    ap = average_precision_score(eval_df[TARGET], eval_df[col])
    axes[1].plot(rec, prec, color=MODEL_COLS[m], lw=lw, linestyle=ls, label=f"{m}  AUC-PR={ap:.3f}")

axes[0].plot([0,1],[0,1],"k:",lw=1,label="No skill")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC — Onset forecast (h=3, 2022–2024)")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].axhline(base_rate, color="k", lw=1, linestyle=":", label=f"No skill ({base_rate:.3f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("PR curve — Onset forecast (h=3, 2022–2024)")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
axes[1].set_ylim(bottom=0)

fig.tight_layout()
fig.savefig(FIG / "e2e_roc_pr.png", dpi=150)
plt.show()
print("Saved → output/figures/e2e_roc_pr.png")

In [14]:
# ── Headline metrics table ────────────────────────────────────────────────────
brier_ref = base_rate * (1 - base_rate)
rows = []
for m in MODELS_EVAL:
    col = f"ons_preds_{m}"
    y, p = eval_df[TARGET].values, eval_df[col].values
    fpr_, tpr_, th_ = roc_curve(y, p)
    tau_j = float(th_[np.argmax(tpr_ - fpr_)])
    y_hat = (p >= tau_j).astype(int)
    tn, fp_c, fn_c, tp = confusion_matrix(y, y_hat).ravel()
    rows.append({
        "Model":       m,
        "AUC-ROC":     round(roc_auc_score(y,p), 3),
        "AUC-PR":      round(average_precision_score(y,p), 3),
        "MCC":         round(matthews_corrcoef(y, y_hat), 3),
        "F1":          round(f1_score(y, y_hat), 3),
        "Sensitivity": round(tp/(tp+fn_c), 3),
        "Specificity": round(tn/(tn+fp_c), 3),
        "BSS":         round(1 - brier_score_loss(y,p) / brier_ref, 3),
        "τ_Youden":    round(tau_j, 4),
    })

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(TBL / "e2e_metrics.csv", index=False)
print(metrics_df.to_string(index=False))
print(f"\nNo-skill AUC-PR baseline: {base_rate:.4f}")
print("Saved → output/tables/e2e_metrics.csv")

Model  AUC-ROC  AUC-PR   MCC    F1  Sensitivity  Specificity    BSS  τ_Youden
   M0    0.527   0.073 0.064 0.156        0.519        0.603 -2.726    0.4857
   M2    0.835   0.293 0.338 0.346        0.779        0.789 -1.638    0.5210
   M3    0.840   0.292 0.303 0.287        0.923        0.650 -1.544    0.4341
   M4    0.846   0.324 0.309 0.303        0.865        0.701 -0.990    0.4016

No-skill AUC-PR baseline: 0.0719
Saved → output/tables/e2e_metrics.csv


---
## Pipeline Summary

| File | Description |
|------|-------------|
| `data/interim/laus_panel_with_shocks.parquet` | 9,180 rows — 51 states × 180 months, UR + Sahm shocks |
| `data/interim/google_trends_panel.parquet` | 55,080 rows — 6 terms × 51 states × 180 months |
| `data/interim/fred_macro.parquet` | 180 rows — national macro controls |
| `data/interim/state_initial_claims.parquet` | 9,180 rows — state weekly claims aggregated monthly |
| `data/interim/state_payrolls.parquet` | 9,180 rows — state nonfarm payrolls (BLS CES) |
| `data/processed/panel_with_features.parquet` | ~8,400 rows — all features with expanding GT z-scores |
| `data/processed/onset_predictions.parquet` | 1,836 rows — onset forecasts M0/M2/M3/M4 (2022–2024) |

**Key results** — onset forecast (h=3, OOS 2022–2024):

| Model | AUC-ROC | AUC-PR | Note |
|-------|---------|--------|------|
| M0 (persistence only) | 0.527 | 0.073 | ~no-skill baseline |
| M2 (+ macro controls) | 0.835 | 0.293 | large gain from macro features |
| M3 (+ state leading) | 0.840 | 0.292 | marginal gain from claims/payrolls |
| M4 (+ Google Trends) | **0.846** | **0.324** | best; GT adds modest but consistent lift |

Adding macro controls (M0→M2) provides by far the largest jump in predictive ability. Google Trends (M3→M4) add a consistent but modest improvement, particularly on precision-recall where the base rate is only ~7%.

---
## BONUS — Decision-Cost Framework

**Who decides?** State labor commissioners and workforce agencies running a Rapid Response Team (RRT).

**The prediction problem:** Each month the model outputs a probability that a Sahm-rule shock will begin in the next 3 months. The decision-maker picks a threshold τ: if p >= τ, activate the RRT; otherwise wait.

**Costs (per state-month decision):**

| Outcome | Interpretation | Cost |
|---------|---------------|------|
| False Positive | Activate RRT, no shock comes | $1M wasted |
| False Negative | Miss the shock, no RRT deployed | $10M social cost |
| True Positive | Correctly activate | $0 (benefit is the avoided FN) |
| True Negative | Correctly stay idle | $0 |

The FN/FP ratio of 10:1 reflects that missing a shock is far more costly than a false alarm.

In [ ]:
from sklearn.metrics import auc

# Cost parameters
Cost_FP = 1    # $M — false alarm cost
Cost_FN = 10   # $M — missed shock cost
Cost_TP = 0
Cost_TN = 0

sub = eval_df.copy()  # eval_df already loaded in Step 7
MODELS_COST = ["M0", "M2", "M3", "M4"]
decisions_per_year = 51 * 12

print(f"Observations: {len(sub):,}  |  Positives: {sub[TARGET].sum()}  |  Base rate: {sub[TARGET].mean():.4f}")
print(f"Cost_FP=${Cost_FP}M  Cost_FN=${Cost_FN}M  (ratio {Cost_FN}:{Cost_FP})")

### ROC Curve with Isocost Line

The optimal threshold τ* is where the isocost line is tangent to the ROC curve. The slope of the isocost line is `(Cost_FP / Cost_FN) × (N/P)`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

y_all = sub[TARGET].values
P_pos = int(y_all.sum())
P_neg = len(y_all) - P_pos
N_over_P = P_neg / P_pos

best_taus = {}
for m in MODELS_COST:
    col   = f"ons_preds_{m}"
    valid = sub.dropna(subset=[col])
    y     = valid[TARGET].values
    p     = valid[col].values

    fpr, tpr, thresholds = roc_curve(y, p)
    roc_auc = auc(fpr, tpr)
    lw = 2.5 if m == "M4" else 1.5
    ax.plot(fpr, tpr, color=MODEL_COLS[m], lw=lw,
            label=f"{m} (AUC={roc_auc:.3f})")

    cost_vals = Cost_FP * fpr * P_neg / len(y) + Cost_FN * (1 - tpr) * P_pos / len(y)
    idx = np.argmin(cost_vals)
    best_taus[m] = float(thresholds[idx]) if idx < len(thresholds) else 0.5
    ax.scatter(fpr[idx], tpr[idx], color=MODEL_COLS[m], s=80, zorder=5)

slope = (Cost_FP / Cost_FN) * N_over_P
fpr_line = np.linspace(0, 1, 100)
tpr_line = slope * fpr_line + (1 - slope * 0.5)
ax.plot(fpr_line, np.clip(tpr_line, 0, 1), "k--", lw=1.2, alpha=0.5,
        label=f"Isocost slope={slope:.2f}")

ax.plot([0, 1], [0, 1], "k:", lw=0.8, alpha=0.4)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curves with isocost line\n(dots = optimal tau* per model)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG / "07_roc_isocost.png", dpi=150)
plt.show()
print("Saved output/figures/07_roc_isocost.png")
print(f"\nOptimal tau* per model: { {m: round(t,3) for m,t in best_taus.items()} }")

### Threshold Sweep — expected cost per decision

For each τ ∈ [0,1], compute total expected cost. The optimal τ* minimises this cost.

In [ ]:
def cost_sweep(y_true, y_prob, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0, 1, 500)
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    rows = []
    for tau in thresholds:
        y_hat = (y_prob >= tau).astype(int)
        tp = int(((y_hat==1)&(y_true==1)).sum())
        fp = int(((y_hat==1)&(y_true==0)).sum())
        fn = int(((y_hat==0)&(y_true==1)).sum())
        tn = int(((y_hat==0)&(y_true==0)).sum())
        cost = (fp * Cost_FP + fn * Cost_FN) / len(y_true)
        rows.append({"tau": tau, "cost": cost, "tp": tp, "fp": fp, "fn": fn, "tn": tn})
    return pd.DataFrame(rows)

sweeps = {}
for m in MODELS_COST:
    col   = f"ons_preds_{m}"
    valid = sub.dropna(subset=[col])
    sweeps[m] = cost_sweep(valid[TARGET], valid[col])

fig, ax = plt.subplots(figsize=(10, 5))
for m in MODELS_COST:
    sw   = sweeps[m]
    best = sw.loc[sw["cost"].idxmin()]
    lw   = 2.5 if m == "M4" else 1.6
    ls   = "-" if m == "M4" else "--"
    ax.plot(sw["tau"], sw["cost"], color=MODEL_COLS[m], lw=lw, ls=ls,
            label=f"{m}  (tau*={best['tau']:.3f}, cost={best['cost']:.3f}$M)")
    ax.axvline(best["tau"], color=MODEL_COLS[m], lw=0.8, alpha=0.4, ls=":")

ax.set_xlabel("Threshold tau"); ax.set_ylabel("Expected cost per decision ($M)")
ax.set_title(f"Cost model: expected cost vs. activation threshold\n"
             f"(Cost_FP=${Cost_FP}M, Cost_FN=${Cost_FN}M)")
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_xlim(0, 1)
fig.tight_layout()
fig.savefig(FIG / "07_cost_sweep.png", dpi=150)
plt.show()
print("Saved output/figures/07_cost_sweep.png")

print(f"\n{'Model':<6}  {'tau*':>6}  {'Cost/decision':>14}  {'TP':>4}  {'FP':>5}  {'FN':>4}")
print("-" * 48)
for m in MODELS_COST:
    sw   = sweeps[m]
    best = sw.loc[sw["cost"].idxmin()]
    print(f"{m:<6}  {best['tau']:6.4f}  {best['cost']:14.4f}  {int(best['tp']):4d}  {int(best['fp']):5d}  {int(best['fn']):4d}")

### Confusion Matrix at τ* (M4)

How many shocks does M4 catch at its optimal threshold? How many false alarms?

In [ ]:
m = "M4"
col   = f"ons_preds_{m}"
valid = sub.dropna(subset=[col])
y     = valid[TARGET].values
p     = valid[col].values

sw       = sweeps[m]
best     = sw.loc[sw["cost"].idxmin()]
tau_star = float(best["tau"])
y_hat    = (p >= tau_star).astype(int)

tp = int(((y_hat==1)&(y==1)).sum())
fp = int(((y_hat==1)&(y==0)).sum())
fn = int(((y_hat==0)&(y==1)).sum())
tn = int(((y_hat==0)&(y==0)).sum())

cm_mat = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm_mat, cmap="Blues")
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["Pred: No shock","Pred: Shock"])
ax.set_yticklabels(["True: No shock","True: Shock"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm_mat[i,j], ha="center", va="center", fontsize=14,
                color="white" if cm_mat[i,j] > cm_mat.max()/2 else "black")
ax.set_title(f"M4 Confusion Matrix at tau*={tau_star:.3f}")
fig.tight_layout()
fig.savefig(FIG / "07_confusion_matrix.png", dpi=150)
plt.show()

print(f"At tau*={tau_star:.3f}:")
print(f"  Shocks caught (TP):     {tp} / {tp+fn}  ({100*tp/(tp+fn):.0f}% recall)")
print(f"  False alarms (FP):      {fp} / {fp+tn}  ({100*fp/(fp+tn):.1f}% FPR)")
print(f"  Missed shocks (FN):     {fn}")
print(f"  Expected cost/decision: ${best['cost']:.3f}M")

### Value of Google Trends (M4 vs M3)

At its optimal threshold, M4 catches 90 out of 104 shocks (87% recall) but sends 402 false alarms — for every real shock it catches, it raises 4-5 false alarms. How much cheaper is M4 compared to M3 at their respective optimal thresholds? This quantifies the dollar value of the text signal.

In [ ]:
results = {}
for m in MODELS_COST:
    sw   = sweeps[m]
    best = sw.loc[sw["cost"].idxmin()]
    results[m] = {
        "tau_star":      round(float(best["tau"]), 4),
        "cost_per_M":    round(float(best["cost"]), 4),
        "cost_annual_M": round(float(best["cost"]) * decisions_per_year, 1),
        "tp": int(best["tp"]), "fp": int(best["fp"]), "fn": int(best["fn"]),
    }

res_df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "model"})
res_df.to_csv(TBL / "07_cost_model_summary.csv", index=False)

m3_cost = results["M3"]["cost_annual_M"]
m4_cost = results["M4"]["cost_annual_M"]
value_of_text = m3_cost - m4_cost

print("=== Cost model summary ===")
print(res_df[["model","tau_star","cost_per_M","cost_annual_M","tp","fp","fn"]].to_string(index=False))
print(f"\nValue of Google Trends signal (M3 cost - M4 cost):")
print(f"  ${value_of_text:+.1f}M/year")
if value_of_text > 0:
    print(f"  M4 saves ${value_of_text:.1f}M/year vs M3 by catching more shocks early.")
else:
    print(f"  M4 is ${abs(value_of_text):.1f}M/year MORE expensive than M3 due to extra false alarms.")
    print(f"  However, M4 misses {results['M4']['fn']} shocks vs {results['M3']['fn']} for M3.")

### Discussion

All models with macro features (M2, M3, M4) reduce expected cost a lot compared to doing nothing. M2 alone gets to about $0.07M per decision, so basic unemployment lags already go a long way.

With Cost_FN/Cost_FP = 10:1, the optimal threshold sits well below 0.5 for all models. You should activate the RRT even at relatively low predicted probabilities because missing a shock is so costly.

M4 uses the lowest threshold (around 0.40) because Google Trends pushes up predicted probabilities before shocks hit. At a 10:1 ratio this means more false alarms than M3, which makes it slightly more expensive overall. But M4 only misses 14 shocks vs 22 for M3.

The sensitivity analysis shows that M4 is actually cheaper than M3 when the FN/FP ratio is low (1:1 to 4:1). Once false negatives cost more than about 5x false alarms, M3 becomes the better choice because M4 generates too many false alarms to compensate.

### Sensitivity Analysis

The cost parameters involve judgement. Here we vary the FN/FP ratio from 1:1 to 20:1 to see how the optimal threshold and total cost change across M3 and M4.

In [ ]:
ratios = np.arange(1, 21, 1)
sens_rows = []

for ratio in ratios:
    cfp = 1; cfn = ratio
    for m in ["M3", "M4"]:
        col   = f"ons_preds_{m}"
        valid = sub.dropna(subset=[col])
        y     = valid[TARGET].values
        p     = valid[col].values
        best_cost, best_tau = np.inf, 0.5
        for tau in np.linspace(0, 1, 300):
            y_hat = (p >= tau).astype(int)
            fp = int(((y_hat==1)&(y==0)).sum())
            fn = int(((y_hat==0)&(y==1)).sum())
            cost = (fp * cfp + fn * cfn) / len(y)
            if cost < best_cost:
                best_cost, best_tau = cost, tau
        sens_rows.append({"ratio": ratio, "model": m,
                          "tau_star": best_tau, "cost_per_M": best_cost,
                          "cost_annual_M": best_cost * decisions_per_year})

sens_df = pd.DataFrame(sens_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
for m, col in [("M3", MODEL_COLS["M3"]), ("M4", MODEL_COLS["M4"])]:
    s = sens_df[sens_df["model"] == m]
    ax.plot(s["ratio"], s["tau_star"], color=col, lw=2, label=m)
ax.axvline(10, color="k", lw=1, ls="--", label="Baseline ratio=10")
ax.set_xlabel("Cost_FN / Cost_FP ratio"); ax.set_ylabel("Optimal threshold tau*")
ax.set_title("Sensitivity: tau* vs. FN/FP ratio")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
for m, col in [("M3", MODEL_COLS["M3"]), ("M4", MODEL_COLS["M4"])]:
    s = sens_df[sens_df["model"] == m]
    ax.plot(s["ratio"], s["cost_annual_M"], color=col, lw=2, label=m)
ax.axvline(10, color="k", lw=1, ls="--", label="Baseline ratio=10")
ax.set_xlabel("Cost_FN / Cost_FP ratio"); ax.set_ylabel("Annual expected cost ($M)")
ax.set_title("Sensitivity: annual cost vs. FN/FP ratio")

m3_vals = sens_df[sens_df["model"]=="M3"].set_index("ratio")["cost_annual_M"]
m4_vals = sens_df[sens_df["model"]=="M4"].set_index("ratio")["cost_annual_M"]
axes[1].fill_between(m4_vals.index, m3_vals, m4_vals,
                     where=(m4_vals <= m3_vals), alpha=0.15, color=MODEL_COLS["M4"],
                     label="M4 cheaper region")
axes[1].legend(fontsize=9)

fig.suptitle("Sensitivity analysis — varying FN/FP cost ratio", fontsize=11)
fig.tight_layout()
fig.savefig(FIG / "07_sensitivity.png", dpi=150)
plt.show()
print("Saved output/figures/07_sensitivity.png")